# API Gateway / Gatekeeper Pattern | Agent Safety & Resilience

In [1]:
# API Gateway for Agent Access Control
import time
from typing import Dict, Optional
from dataclasses import dataclass, field

In [2]:
@dataclass
class GatewayConfig:
    rate_limit_per_minute: int = 10
    valid_api_keys: set = field(default_factory=lambda: {"key-123", "key-456"})

class AgentGateway:
    def __init__(self, config: GatewayConfig):
        self.config = config
        self._request_log: list = []
        self._rate_counters: Dict[str, list] = {}

    def authenticate(self, api_key: str) -> bool:
        return api_key in self.config.valid_api_keys

    # Request schema validation
    REQUIRED_FIELDS = {"query", "agent_name"}

    def validate_schema(self, request: dict) -> tuple:
        missing = self.REQUIRED_FIELDS - set(request.keys())
        if missing:
            return False, f"Missing required fields: {missing}"
        if len(str(request.get("query", ""))) > 1000:
            return False, "Query exceeds 1000 character limit"
        return True, "OK"

    def check_rate_limit(self, api_key: str) -> bool:
        now = time.time()
        requests = [t for t in self._rate_counters.get(api_key, []) if now - t < 60]
        self._rate_counters[api_key] = requests
        if len(requests) >= self.config.rate_limit_per_minute:
            return False
        self._rate_counters[api_key].append(now)
        return True

    def process_request(self, api_key: str, request: dict) -> dict:
        # Schema validation
        valid, msg = self.validate_schema(request)
        if not valid:
            return {"status": 400, "body": msg}
        # Authentication
        if not self.authenticate(api_key):
            return {"status": 401, "body": "Unauthorized: Invalid API key"}
        # Rate limiting
        if not self.check_rate_limit(api_key):
            return {"status": 429, "body": "Rate limit exceeded"}
        # Log and forward
        query = request.get("query", "")
        self._request_log.append({"key": api_key, "request": query[:100], "time": time.time()})
        # In production, this would forward to the actual agent
        return {"status": 200, "body": f"Agent response for: {query[:50]}..."}

In [3]:
gateway = AgentGateway(GatewayConfig(rate_limit_per_minute=3))
print(gateway.process_request("key-123", {"query": "What is the weather?", "agent_name": "weather_bot"}))
print(gateway.process_request("invalid-key", {"query": "Hack", "agent_name": "x"}))
print(gateway.process_request("key-123", {"query": "Query 2", "agent_name": "bot"}))
print(gateway.process_request("key-123", {"query": "Query 3", "agent_name": "bot"}))
print(gateway.process_request("key-123", {"query": "Query 4", "agent_name": "bot"}))  # Rate limited

# Test: schema validation failure
print(gateway.process_request("key-123", {"query": "Missing agent_name field"}))  # Missing required field

{'status': 200, 'body': 'Agent response for: What is the weather?...'}
{'status': 401, 'body': 'Unauthorized: Invalid API key'}
{'status': 200, 'body': 'Agent response for: Query 2...'}
{'status': 200, 'body': 'Agent response for: Query 3...'}
{'status': 429, 'body': 'Rate limit exceeded'}
{'status': 400, 'body': "Missing required fields: {'agent_name'}"}
